# 07 — NRC VAD v2.1: General Lexical Affect

Loads **NRC VAD v2.1** — 54,801 word and phrase entries rated on three dimensions.
Only **Valence** is used for downstream SCL scoring; Arousal and Dominance
are kept for reference.

**Source:** `data/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt`  
**Scale:** bipolar [-1, +1] (0 ≈ neutral)  
**Columns kept:** all (`term`, `vad_valence`, `vad_arousal`, `vad_dominance`)


In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

VAD_FILE = Path('data/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt')


## Load

In [2]:
vad = pd.read_csv(VAD_FILE, sep='\t')
vad.columns = ['term', 'vad_valence', 'vad_arousal', 'vad_dominance']

print(f'Entries       : {len(vad):,}')
print(f'Valence range : [{vad["vad_valence"].min():.3f}, {vad["vad_valence"].max():.3f}]')
print(f'Arousal range : [{vad["vad_arousal"].min():.3f}, {vad["vad_arousal"].max():.3f}]')
print(f'Dominance range: [{vad["vad_dominance"].min():.3f}, {vad["vad_dominance"].max():.3f}]')
print(f'Columns       : {list(vad.columns)}')
vad.head(5)


Entries       : 54,801
Valence range : [-1.000, 1.000]
Arousal range : [-1.000, 1.000]
Dominance range: [-1.000, 1.000]
Columns       : ['term', 'vad_valence', 'vad_arousal', 'vad_dominance']


,term,vad_valence,vad_arousal,vad_dominance
0,a battery,0.134,-0.298,-0.096
1,a bit,-0.096,-0.264,-0.214
2,a bunch,0.088,-0.350,-0.068
3,a cappella,0.134,-0.116,-0.200
4,a couple,0.266,-0.110,0.090


## Valence distribution

VAD valence is bipolar [-1, +1], same scale as OPP and NMA.

In [3]:
fig = px.histogram(vad, x='vad_valence', nbins=50,
    title='NRC VAD v2.1 — Valence distribution',
    labels={'vad_valence': 'Valence [-1, +1]'})
fig.add_vline(x=0, line_dash='dash', line_color='grey', annotation_text='neutral')
fig.show()

print(vad['vad_valence'].describe().round(4).to_string())
print(f'\nPositive (>0.5) : {(vad["vad_valence"] > 0).sum():,}')
print(f'Negative (<0) : {(vad["vad_valence"] < 0).sum():,}')
print(f'Neutral  (=0) : {(vad["vad_valence"] == 0).sum():,}')


count    54801.0000
mean        -0.0014
std          0.4766
min         -1.0000
25%         -0.3340
50%          0.0000
75%          0.3330
max          1.0000

Positive (>0.5) : 25,358
Negative (<0) : 23,520
Neutral  (=0) : 5,923


In [4]:
import plotly.graph_objects as go
fig2 = go.Figure()
for col, label in [('vad_valence','Valence'),('vad_arousal','Arousal'),('vad_dominance','Dominance')]:
    fig2.add_trace(go.Histogram(x=vad[col], name=label, opacity=0.6, nbinsx=50))
fig2.update_layout(barmode='overlay',
    title='NRC VAD v2.1 — all three dimensions',
    xaxis_title='Score [0, 1]', yaxis_title='Count')
fig2.show()


## N-gram breakdown

In [5]:
vad['ngrams'] = vad['term'].str.split().str.len()
counts = vad['ngrams'].value_counts().sort_index()
counts.index = counts.index.map(lambda n: {1:'unigram',2:'bigram',3:'trigram'}.get(n,f'{n}-gram'))
print(counts.to_string())

fig3 = px.box(vad, x=vad['ngrams'].map(lambda n: {1:'unigram',2:'bigram',3:'trigram'}.get(n,'4+')),
    y='vad_valence', points='outliers',
    title='NRC VAD v2.1 — valence by n-gram length',
    labels={'x':'n-gram type','vad_valence':'Valence'})
fig3.show()


ngrams
unigram    44727
bigram     10071
trigram        2


## Most positive / most negative

In [6]:
print('Most positive:')
display(vad.nlargest(10, 'vad_valence')[['term','vad_valence','vad_arousal','vad_dominance']])
print('\nMost negative:')
display(vad.nsmallest(10, 'vad_valence')[['term','vad_valence','vad_arousal','vad_dominance']])


Most positive:


,term,vad_valence,vad_arousal,vad_dominance
91,abilities,1.0,0.056,0.708
184,absolver,1.0,0.000,0.500
223,abundantly,1.0,0.000,0.286
286,acclamation,1.0,0.667,0.593
378,achievable,1.0,-0.185,0.556
380,achieve success,1.0,0.118,0.750
436,acquitted,1.0,-0.556,0.542
437,acquitter,1.0,0.583,0.733
450,acrobatic,1.0,1.000,0.519
485,activeness,1.0,1.000,0.708



Most negative:


,term,vad_valence,vad_arousal,vad_dominance
49,abase,-1.0,0.556,-0.958
50,abased,-1.0,-0.667,-0.917
51,abasement,-1.0,-0.333,-0.958
76,abductor,-1.0,0.714,0.833
85,abhorrently,-1.0,-0.074,0.074
86,abhorring,-1.0,0.704,0.833
95,abjection,-1.0,-0.667,-0.952
97,abjectness,-1.0,-0.762,-1.000
120,abominableness,-1.0,-0.667,-0.593
121,abominably,-1.0,-0.583,-0.611
